In [1]:
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import DBSCAN
# from scipy.spatial.distance import cdist
# import os

# file_path = r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx'
# output_filename2 = r'outputs\G11\dsas_g11_generator_bearings_correlation_detection\correlation_detection\dsas_g11_generator_bearings_correlation_detection_output2.xlsx'
# all_features = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
# target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']

# # ۲. خواندن فایل اصلی
# try:
#     df_raw = pd.read_excel(file_path)
#     df_raw['date'] = pd.to_datetime(df_raw['date'])
#     df_raw = df_raw.sort_values(by='date')
#     print("فایل ورودی با موفقیت خوانده شد.")
# except Exception as e:
#     print(f"خطا در خواندن فایل: {e}")
#     exit()

# # ۳. پیش‌پردازش و حذف ۱۰ درصد داده‌های پرت (DBSCAN)
# scaler = StandardScaler()
# scaled_data = scaler.fit_transform(df_raw[all_features])

# dbscan = DBSCAN(eps=0.5, min_samples=5)
# labels = dbscan.fit_predict(scaled_data)

# cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}

# def calculate_distance(i):
#     label, point = labels[i], scaled_data[i].reshape(1, -1)
#     if label != -1:
#         return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
#     return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0

# df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
# df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
# df_cleaned = df_cleaned.sort_values(by='date').set_index('date')

# # ۴. بازه ۳۰ روز آخر
# last_date = df_cleaned.index.max()
# start_analysis_date = last_date - pd.Timedelta(days=30)
# print(f"شروع تحلیل Rolling از تاریخ: {start_analysis_date}")

# # ۵. محاسبه Rolling Correlation و ذخیره در لیست‌ها
# results_list = [] # برای خروجی شماره ۱

# print("در حال محاسبه ماتریس کوریلیشن متحرک...")

# # محاسبه کوریلیشن برای تمامی جفت‌ها به صورت یکجا برای بهینه‌سازی
# for target in target_sensors:
#     # ایجاد یک دیکشنری برای ذخیره مقادیر این تارگت در هر لحظه (برای خروجی ۲)
#     # این دیکشنری در نهایت به دیتافریم تبدیل می‌شود
#     temp_storage = {}

#     for feature in all_features:
#         if target == feature:
#             # کوریلیشن با خودش را NaN در نظر می‌گیریم
#             temp_storage[feature] = np.nan
#             continue

#         # محاسبه Rolling Correlation
#         rolling_series = df_cleaned[target].rolling(window='30D').corr(df_cleaned[feature])
#         # فقط بازه ۳۰ روز آخر را نگه می‌داریم
#         temp_storage[feature] = rolling_series[rolling_series.index >= start_analysis_date]


#     # ساخت ردیف‌های خروجی ۲ برای این تارگت خاص
#     target_df_wide = pd.DataFrame(temp_storage)
#     target_df_wide['AssetID'] = target
#     target_df_wide = target_df_wide.reset_index()

#     # مرتب‌سازی ستون‌ها: date, AssetID و سپس بقیه
#     cols = ['date', 'AssetID'] + all_features
#     results_list.append(target_df_wide[cols])

# # ۶. ترکیب نتایج و ساخت فایل‌های خروجی
# df_output2 = pd.concat(results_list, ignore_index=True)

# # تولید خروجی شماره ۱ از روی خروجی شماره ۲ (تبدیل Wide به Long)
# # این کار باعث می‌شود محاسبات دوباره تکرار نشود و هر دو فایل با هم منطبق باشند
# df_output1 = df_output2.melt(id_vars=['date', 'AssetID'], 
#                              value_vars=all_features, 
#                              var_name='AssetID_correlation', 
#                              value_name='correlation_value')

# # حذف مقادیر خودش با خودش (NaN) در خروجی ۱
# df_output1 = df_output1.dropna(subset=['correlation_value'])

# # ۷. ذخیره نهایی
# try:
#     # ذخیره خروجی ۲ (ساختار ستونی - درخواستی جدید)
#     df_output2.to_excel(output_filename2, index=False)

#     print("\nعملیات با موفقیت پایان یافت.")
#     print(f"فایل ۲ (ساختار ستونی): {output_filename2}")
#     print(f"تعداد ردیف‌های خروجی نهایی: {len(df_output2)}")
# except Exception as e:
#     print(f"خطا در ذخیره فایل‌ها: {e}")

# # نمایش نمونه خروجی ۲
# print("\nنمونه ساختار فایل شماره ۲:")
# print(df_output2.head())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import cdist
import os
import time
from datetime import datetime

# غیرفعال کردن هشدارهای غیرضروری
import warnings
warnings.filterwarnings('ignore')

def run_correlation_analysis():
    """اجرای تحلیل همبستگی برای ژنراتور و ذخیره خروجی (فقط Wide)"""
    
    print("="*60)
    print(f"🔄 شروع تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)
    
    # ۱. تنظیمات مسیرها و متغیرها
    file_path = r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx'
    output_filename2 = r'outputs\G11\dsas_g11_generator_bearings_correlation_detection\correlation_detection\dsas_g11_generator_bearings_correlation_detection_output2.xlsx'
    
    # ایجاد پوشه خروجی
    os.makedirs(os.path.dirname(output_filename2), exist_ok=True)
    
    all_features = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 
                    'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
    
    target_sensors = all_features

    # ۲. خواندن فایل اصلی
    try:
        df_raw = pd.read_excel(file_path)
        df_raw['date'] = pd.to_datetime(df_raw['date'])
        df_raw = df_raw.sort_values(by='date')
        print(f"✅ فایل ورودی با موفقیت خوانده شد. تعداد رکوردها: {len(df_raw):,}")
        print(f"📅 بازه زمانی: {df_raw['date'].min()} تا {df_raw['date'].max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return None

    # ۳. پیش‌پردازش و حذف ۱۰ درصد داده‌های پرت (DBSCAN)
    print("🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...")
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_raw[all_features])

    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels = dbscan.fit_predict(scaled_data)

    cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}

    def calculate_distance(i):
        label, point = labels[i], scaled_data[i].reshape(1, -1)
        if label != -1:
            return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
        return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0

    before_count = len(df_raw)
    df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
    df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
    df_cleaned = df_cleaned.sort_values(by='date').set_index('date')
    after_count = len(df_cleaned)
    
    print(f"   حذف {before_count - after_count:,} ردیف به عنوان داده‌های پرت")

    # ۴. بازه ۳۰ روز آخر
    last_date = df_cleaned.index.max()
    start_analysis_date = last_date - pd.Timedelta(days=30)
    print(f"📅 شروع تحلیل Rolling از تاریخ: {start_analysis_date}")

    # ۵. محاسبه Rolling Correlation و ذخیره در لیست‌ها
    print("🔄 مرحله 2: محاسبه ماتریس کوریلیشن متحرک...")
    
    results_list = []

    # محاسبه کوریلیشن برای تمامی جفت‌ها به صورت یکجا برای بهینه‌سازی
    for target in target_sensors:
        temp_storage = {}

        for feature in all_features:
            if target == feature:
                temp_storage[feature] = np.nan
                continue

            # محاسبه Rolling Correlation
            rolling_series = df_cleaned[target].rolling(window='30D').corr(df_cleaned[feature])
            temp_storage[feature] = rolling_series[rolling_series.index >= start_analysis_date]

        # ساخت ردیف‌های خروجی ۲ برای این تارگت خاص
        target_df_wide = pd.DataFrame(temp_storage)
        target_df_wide['AssetID'] = target
        target_df_wide = target_df_wide.reset_index()

        cols = ['date', 'AssetID'] + all_features
        results_list.append(target_df_wide[cols])

    # ۶. ترکیب نتایج و ساخت فایل‌های خروجی
    df_output2 = pd.concat(results_list, ignore_index=True)
    print(f"   ✅ خروجی همبستگی (Wide): {len(df_output2):,} رکورد")

    # ۷. ذخیره نهایی
    print("💾 مرحله 3: ذخیره خروجی...")
    
    try:
        df_output2.to_excel(output_filename2, index=False)

        print(f"\n✅ عملیات با موفقیت پایان یافت.")
        print(f"📂 فایل (ساختار ستونی - Wide): {output_filename2}")
        print(f"📊 تعداد رکوردهای نهایی: {len(df_output2):,}")
        print(f"📋 تعداد ستون‌ها: {len(df_output2.columns)}")
        
        # نمایش نمونه خروجی
        print("\n📋 نمونه ساختار فایل:")
        print(df_output2.head())
        
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل‌ها: {e}")
        return None
    
    print("="*60)
    print(f"✅ تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} کامل شد")
    print("="*60)
    
    return df_output2

def run_scheduler():
    """
    بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز)
    """
    print("="*60)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - تحلیل همبستگی ژنراتور (فقط Wide)")
    print("="*60)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 10:00")
    print("   - ساعت 10:05")
    print("   - ساعت 10:10")
    print("="*60)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*60)
    
    last_run_time = None  # فقط برای جلوگیری از اجرای مجدد در یک زمان
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            # بررسی زمان‌های مشخص
            if current_time in ["23:44", "23:45", "23:46"]:
                # فقط چک می‌کنیم که در همین زمان دوبار اجرا نشود
                if last_run_time != current_time:
                    print("\n" + "="*60)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*60)
                    
                    # اجرای تابع اصلی
                    result = run_correlation_analysis()
                    
                    if result is not None:
                        print("\n" + "="*60)
                        print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                        print("="*60)
                    else:
                        print("\n" + "="*60)
                        print("❌ اجرای زمان‌بندی شده با شکست مواجه شد!")
                        print("="*60)
                    
                    # ثبت زمان اجرا
                    last_run_time = current_time
                    
                    # 10 ثانیه صبر کن تا از اجرای مجدد در همان دقیقه جلوگیری شود
                    time.sleep(10)
            
            # هر 10 ثانیه یکبار بررسی کن
            time.sleep(10)
            
        except KeyboardInterrupt:
            print("\n" + "="*60)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*60)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            print("🔄 ادامه اجرا...")
            time.sleep(60)

# اجرای اصلی
if __name__ == "__main__":
    try:
        print("="*60)
        print("🚀 شروع برنامه تحلیل همبستگی ژنراتور (فقط Wide)")
        print("="*60)
        
        # شروع زمان‌بندی
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        input("برای خروج Enter بزنید...")

🚀 شروع برنامه تحلیل همبستگی ژنراتور (فقط Wide)
🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - تحلیل همبستگی ژنراتور (فقط Wide)
⏰ زمان‌های اجرا (هر روز):
   - ساعت 10:00
   - ساعت 10:05
   - ساعت 10:10
💡 برای توقف برنامه، Ctrl+C را بزنید

⏰ زمان اجرا فرا رسید: 2026-07-01 23:44:06
🔄 شروع تحلیل در 2026-07-01 23:44:06
✅ فایل ورودی با موفقیت خوانده شد. تعداد رکوردها: 11,915
📅 بازه زمانی: 2021-03-16 05:33:48 تا 2026-05-31 20:30:17
🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...
   حذف 1,191 ردیف به عنوان داده‌های پرت
📅 شروع تحلیل Rolling از تاریخ: 2026-05-01 20:30:17
🔄 مرحله 2: محاسبه ماتریس کوریلیشن متحرک...
   ✅ خروجی همبستگی (Wide): 1,575 رکورد
💾 مرحله 3: ذخیره خروجی...

✅ عملیات با موفقیت پایان یافت.
📂 فایل (ساختار ستونی - Wide): outputs\G11\dsas_g11_generator_bearings_correlation_detection\correlation_detection\dsas_g11_generator_bearings_correlation_detection_output2.xlsx
📊 تعداد رکوردهای نهایی: 1,575
📋 تعداد ستون‌ها: 11

📋 نمونه ساختار فایل:
                 date       AssetID  AssetID_9362 